# 03 - Treino E1 e E2

Treina o YOLO26n em cada base com a mesma configuração (`configs/treino.yaml`,
[detalhamento](../docs/fases/fase-3.md)). **Os conjuntos de teste não são usados.**

**No Colab, com GPU** (T4):

1. Execute tudo e autorize o Google Drive. O E2, menor, roda primeiro.
2. A execução **para** antes do E1: veja o tempo por época, mude `CONFIRMADO_E1` para `True`
   e execute daquela célula em diante.
3. No fim, baixe `epi-sistema/f3/entrega.zip` do Drive.

Se a sessão cair, execute tudo de novo: o que terminou é pulado e o treino interrompido é
retomado da última cópia no Drive. Um experimento não troca de tipo de GPU no meio.

Fora do Colab o padrão é o modo **curto** (2 épocas, só para conferir o código). Para o treino
completo na máquina local: baixe o pacote do Drive para `data/bases_preparadas.tar` e defina
`MODO = "completo"`. E1 e E2 usam o mesmo tipo de GPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Guilh3rme/epi-sistema.git"

if "COLAB_RELEASE_TAG" in os.environ:
    raiz = Path("/content/epi-sistema")
    if not raiz.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(raiz)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))  # dispensa reiniciar o ambiente
else:
    raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

(raiz / "runs").mkdir(exist_ok=True)
os.chdir(raiz / "runs")  # pesos e resultados ficam em runs/, fora do git
print("raiz:", raiz.name)

In [ ]:
from epi.ambiente import configurar_ultralytics, em_colab

configurar_ultralytics(raiz)  # antes de qualquer importação do ultralytics
import shutil

import torch

from epi import dados, treino

MODO = os.environ.get("EPI_MODO") or ("completo" if em_colab() else "curto")
assert MODO in ("curto", "completo"), MODO
COMPLETO = MODO == "completo"
cfg = dados.carregar_config(raiz / "configs" / "treino.yaml")
saida = raiz / "runs" / ("f3" if COMPLETO else "f3_curto")
copia = saida / "copia"
pacote = raiz / "data" / "bases_preparadas.tar"  # plano B local: baixado do Drive
if COMPLETO:
    assert torch.cuda.is_available(), "selecione um ambiente com GPU"
    preparado, divisoes = raiz / "data" / "preparado", raiz / "configs" / "divisoes"
    if em_colab():
        from google.colab import drive

        drive.mount("/content/drive")
        meu_drive = Path("/content/drive/MyDrive/epi-sistema")
        pacote, copia = meu_drive / "f2" / "bases_preparadas.tar", meu_drive / "f3"
else:  # dados do modo curto do notebook 01
    preparado, divisoes = raiz / "data" / "preparado_curto", raiz / "runs" / "f2_curto" / "divisoes"
    cfg["treino"] |= {"epochs": 2, "close_mosaic": 1}
    cfg["copiar_a_cada"] = 1
copia.mkdir(parents=True, exist_ok=True)
peso = raiz / "runs" / "pesos" / "yolo26n.pt"
device = 0 if torch.cuda.is_available() else "cpu"
commit = subprocess.run(["git", "-C", str(raiz), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"MODO: {MODO} | commit: {commit} | CPUs: {os.cpu_count()} | "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")

## Bases preparadas e divisões congeladas

In [ ]:
import time

if COMPLETO and not all((preparado / f"{b}.json").exists() for b in ("sh17", "cppe")):
    inicio = time.time()
    dados.restaurar(pacote, preparado.parent)
    print(f"bases restauradas em {(time.time() - inicio) / 60:.1f} min")
dados.verificar_divisoes(divisoes)
print("divisões conferidas")
print(f"disco livre: {shutil.disk_usage(raiz).free / 1e9:.1f} GB | "
      f"livre no destino das cópias: {shutil.disk_usage(copia).free / 1e9:.1f} GB")

## E2: Construction-PPE

In [ ]:
def rodar(experimento):
    r = treino.treinar(experimento, cfg, preparado, divisoes, saida, copia, peso, device=device)
    print(f"RESUMO {r['nome']}: {r['epocas_executadas']} épocas, melhor {r['melhor_epoca']}, "
          f"mosaico desligado no fim: {r['mosaico_desligado_no_fim']}")
    print(f"{'desenvolvimento':14} {'P':>6} {'R':>6} {'mAP50':>6} {'50-95':>6}")
    for nome, m in {"geral": r["metricas_desenvolvimento"]["geral"],
                    **r["metricas_desenvolvimento"]["por_classe"]}.items():
        print(f"{nome:14} {m['precisao']:6.3f} {m['revocacao']:6.3f} "
              f"{m['map50']:6.3f} {m['map50_95']:6.3f}")
    return r


e2 = rodar("e2")

## PARADA antes do E1

O E1 tem cerca de 8 vezes mais imagens que o E2. Confira o tempo por época acima (linhas
`TEMPO`) antes de continuar. Se o Colab não der conta, o plano B é treinar o E1 na máquina
local.

In [ ]:
CONFIRMADO_E1 = False  # <- mude para True depois de ver o tempo por época
assert CONFIRMADO_E1 or not COMPLETO, "PARADA: confira o tempo por época e confirme o E1."
gpu_e2 = e2["sessoes"][-1]["gpu"]
gpu_atual = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
assert gpu_atual == gpu_e2, f"E1 e E2 usam o mesmo tipo de GPU: E2 usou {gpu_e2}, agora {gpu_atual}"
print("E1 confirmado")

## E1: SH17

In [ ]:
e1 = rodar("e1")
gpus = {s["gpu"] for r in (e1, e2) for s in r["sessoes"]}
print("GPUs usadas:", gpus, "| mesma nos dois" if len(gpus) == 1 else "| ATENÇÃO: GPUs diferentes")

## Entrega

Resumo, curva e melhor peso de cada experimento concluído. Pode ser executada com apenas um
concluído.

In [ ]:
arquivo = treino.entregar(copia, saida / "entrega.zip")
shutil.copyfile(arquivo, copia / "entrega.zip")
print("entrega:", copia / "entrega.zip")

In [ ]:
if COMPLETO and em_colab():
    drive.flush_and_unmount()  # garante o envio ao Drive antes de desconectar
    print("Drive sincronizado: já pode desconectar")
print("treino OK" if COMPLETO else "teste curto OK (os pesos gerados não valem)")